# Day 23 — Variance and standard deviation

This notebook covers Measures of Dispersion, Sample vs. Population Variance, Bessel's Correction ($n-1$), Empirical Rule, and Feature Scaling ($Z$-score Standardization)

## Validation goals
- Understand why central tendency metrics alone fail to capture operational risk or volatility
- Calculate Population Variance ($\sigma^2$) and Sample Variance ($s^2$) manually and using library built-ins
- Explain Bessel's Correction ($n - 1$) and its role in removing downward bias in sample variance estimation
- Interpret Standard Deviation ($\sigma$, $s$) in original units and apply the Empirical Rule (68-95-99.7 Rule)
- Demonstrate how mean and standard deviation enable $Z$-score feature standardization ($x_{\text{scaled}} = \frac{x - \mu}{\sigma}$) for gradient descent stabilization

While measures of central tendency (mean, median, mode) tell you where the center of a dataset lies, they reveal nothing about how spread out or dispersed the data points are. Two datasets can have the exact same mean yet possess completely different risk profiles, volatilities, or error margins.

Measures of dispersion quantify this spread. Variance and standard deviation are the most fundamental metrics used in statistics, machine learning feature scaling (e.g., Z-score standardization), and risk management.

### Core Concepts & Deep Dive

1. **Why Central Tendency Isn't Enough**

Imagine two machine learning model deployment pipelines measuring prediction error in milliseconds:

Pipeline A: `[48, 50, 52, 49, 51]` (Mean = 50ms, highly consistent)

Pipeline B: `[10, 20, 50, 80, 90]` (Mean = 50ms, highly volatile and unpredictable)

Both pipelines share a mean of 50ms, but Pipeline B carries a massive operational risk due to high dispersion. Variance and standard deviation capture this exact difference.

2. **Variance ($\sigma^2$ or $s^2$)**

Variance measures the average squared distance between each data point and the mean.To calculate variance:
- Find the mean ($\bar{x}$) of the dataset.
- Subtract the mean from each individual data point to find its deviation `($x_i - \bar{x}$)`.
- Square each deviation so that positive and negative differences do not cancel each other out.Sum all the squared deviations and divide.
- Population vs. Sample Variance:
    - Population Variance ($\sigma^2$): Used when you have data for the entire population. You divide the sum of squared deviations by the total number of observations ($N$).$$\sigma^2 = \frac{\sum_{i=1}^{N} (x_i - \mu)^2}{N}$$
    - Sample Variance ($s^2$): Used when your dataset is a sample drawn from a larger population. You divide by $n - 1$ instead of $n$. This adjustment is known as Bessel's Correction, which corrects the bias in estimating the population variance from a limited sample.$$s^2 = \frac{\sum_{i=1}^{n} (x_i - \bar{x})^2}{n - 1}$$

3. **Standard Deviation ($\sigma$ or $s$)**

Because variance squares the deviations, its units are also squared (e.g., if measuring revenue in dollars, variance is in dollars squared). This makes variance difficult to interpret intuitively.The standard deviation is simply the square root of the variance, bringing the metric back to the original unit of measurement:$$\sigma = \sqrt{\sigma^2}, \quad s = \sqrt{s^2}$$
- Empirical Rule (68-95-99.7 Rule): For data that follows a normal (Gaussian) distribution:
    - ~68% of data falls within $\pm 1$ standard deviation of the mean.
    - ~95% falls within $\pm 2$ standard deviations.
    - ~99.7% falls within $\pm 3$ standard deviations.   

### Calculating Variance & Standard Deviation

Here is a script demonstrating manual calculation alongside optimized NumPy and Pandas functions, highlighting the difference between population (ddof=0) and sample (ddof=1) degrees of freedom.


In [1]:
import numpy as np
import pandas as pd

# Sample dataset representing server memory utilization percentages
memory_usage = np.array([45.2, 50.1, 48.6, 52.3, 47.9, 49.0, 51.4, 46.8])

df = pd.DataFrame({"memory_pct": memory_usage})

# ==========================================
# 1. Manual Sample Variance & Std Deviation Calculation
# ==========================================
n = len(memory_usage)
mean_val = np.mean(memory_usage)

# Sum of squared deviations
squared_deviations = (memory_usage - mean_val) ** 2
sum_sq_dev = np.sum(squared_deviations)

# Bessel's correction (n - 1) for sample variance
manual_sample_variance = sum_sq_dev / (n - 1)
manual_sample_std = np.sqrt(manual_sample_variance)

print(f"--- Manual Calculations ---")
print(f"Sample Variance:         {manual_sample_variance:.4f}")
print(f"Sample Standard Dev:     {manual_sample_std:.4f}")

# ==========================================
# 2. NumPy & Pandas Built-In Functions
# ==========================================
# Note: NumPy defaults to ddof=0 (population), Pandas defaults to ddof=1 (sample)
np_pop_std = np.std(memory_usage, ddof=0)
np_sample_std = np.std(memory_usage, ddof=1)

pd_sample_std = df["memory_pct"].std() # uses ddof=1 by default

print(f"\n--- Built-in Library Results ---")
print(f"NumPy Population Std Dev (ddof=0): {np_pop_std:.4f}")
print(f"NumPy Sample Std Dev (ddof=1):     {np_sample_std:.4f}")
print(f"Pandas Sample Std Dev:             {pd_sample_std:.4f}")

--- Manual Calculations ---
Sample Variance:         5.4927
Sample Standard Dev:     2.3436

--- Built-in Library Results ---
NumPy Population Std Dev (ddof=0): 2.1923
NumPy Sample Std Dev (ddof=1):     2.3436
Pandas Sample Std Dev:             2.3436


### Guided Question:

If you simply sum the raw deviations from the mean ($\sum (x_i - \bar{x})$) without squaring them, the total sum is mathematically guaranteed to equal zero.Why does squaring the deviations solve this cancellation problem, and what downstream mathematical side-effect does squaring introduce regarding units?

### Answer:

1. **Why squaring prevents cancellation?**
Each deviation is the difference between a data point and the mean.

Values above the mean produce positive deviations, while values below the mean produce negative deviations. When you add them together, the positive and negative values cancel out, so their sum is always zero (apart from possible floating-point rounding errors).

For example, consider the values `[2, 4, 6]`, whose mean is `4`.

Squaring makes every deviation non-negative, so the deviations no longer cancel each other out. Larger deviations also contribute disproportionately more to the variance.

2. **What side effect does squaring introduce?**
Squaring changes the units of measurement to squared units.

In the server-memory example, the original measurements are percentages (%).
- Deviations are measured in percentage points.
- Squared deviations are measured in percentage-points squared.
- Variance is therefore expressed in squared percentage points.
- Standard deviation takes the square root of variance, returning to the original unit: percentage points.

This is why standard deviation is often easier to interpret than variance: it describes spread using the same unit as the original measurements.

Key takeaway: Squaring prevents positive and negative deviations from cancelling, but it makes variance harder to interpret directly because its units are squared. Taking the square root produces the standard deviation, restoring the original measurement unit.

### Guided Question:

When calculating sample variance, we divide by $n - 1$ instead of the total sample size $n$.Why does dividing by $n$ systematically underestimate the true variance of the underlying population, and how does subtracting 1 correct this bias?

### Answer:

The key idea behind Bessel's correction is that when we calculate variance from a sample, we first estimate the population mean from that same sample. This makes the deviations appear slightly smaller than they would if we knew the true population mean.

1. **Why dividing by \(n\) underestimates population variance**
Suppose you take a sample of server memory usage measurements.

The population variance is calculated using the true population mean, \(\mu\):
\[
\sigma^2=\frac{\sum_{i=1}^{N}(x_i-\mu)^2}{N}
\]

But in practice, you usually don't know \(\mu\). You estimate it using the sample mean, \(\bar{x}\).

Because the sample mean is calculated from the same observations, the deviations from it must sum to zero. This constraint means the deviations are not all independent, and the resulting squared deviations tend to be smaller on average than squared deviations from the true population mean.

If you divide their sum by \(n\), the sample variance estimator is biased downward:
\[
E\left[\frac{\sum_{i=1}^{n}(x_i-\bar{x})^2}{n}\right]
=\frac{n-1}{n}\sigma^2
\]

In other words, its expected value is only \(\frac{n-1}{n}\) of the true population variance, assuming independent observations from a population with finite variance.

2. **How dividing by \(n-1\) corrects the bias**
Bessel's correction uses \(n-1\) instead of \(n\):
\[
s^2=\frac{\sum_{i=1}^{n}(x_i-\bar{x})^2}{n-1}
\]

Why does this work? Because the expected sum of squared deviations around the sample mean is \((n-1)\sigma^2\). Dividing by \(n-1\) therefore gives an estimator whose expected value equals the true population variance:
\[
E[s^2]=\sigma^2
\]

This is what unbiased estimation means: across many independent samples, the average estimated variance equals the true population variance.
3. Why \(n-1\) is related to degrees of freedom
Once you know the sample mean, only \(n-1\) deviations can vary independently. The final deviation is constrained by the fact that all deviations must sum to zero.

For example, if three deviations are \(-2\), \(+1\), and an unknown value, that final deviation must be \(+1\). It cannot take an arbitrary value.

This is why we say the sample has \(n-1\) degrees of freedom when estimating variance around its sample mean.

**Practical Python connection:**

In [ ]:
np.var(memory_usage, ddof=0)  # Divide by n: population-style variance
np.var(memory_usage, ddof=1)  # Divide by n-1: unbiased sample variance

NumPy defaults to `ddof=0`, while Pandas `.var()` defaults to `ddof=1`.

Key takeaway: Dividing by \(n\) underestimates population variance on average because the sample mean is estimated from the same data. Dividing by \(n-1\) compensates for that effect and produces an unbiased estimate of the population variance.

### Guided QUestion:

In machine learning, features with vastly different scales (e.g., house square footage ranging from 500 to 5,000 versus number of bedrooms ranging from 1 to 5) can distort gradient descent optimization.How do variance and standard deviation factor directly into Z-score standardization ($x_{\text{scaled}} = \frac{x - \mu}{\sigma}$), and why does this process stabilize model training?

### Answer:

Z-score standardization uses the mean to center a feature around zero and the standard deviation to scale its spread. This helps features with different numerical scales contribute more appropriately during model training.
1. How mean and standard deviation work together
- Mean (\(\mu\)) — centering: Subtracting the mean makes the feature's average approximately zero.
- Standard deviation (\(\sigma\)) — scaling: Dividing by the standard deviation expresses each value in units of standard deviations from the mean.
For example, suppose a house has 2,000 square feet, the dataset's mean is 2,000 square feet, and its standard deviation is 1,000 square feet.

Its standardized value is \(z=0\), meaning it is exactly at the mean.

For a house with 3,000 square feet:
\[
z=\frac{3000-2000}{1000}=1
\]

Its size is one standard deviation above the mean.
2. Why this stabilizes gradient descent
Suppose your model uses these two features:
Feature	Typical range
House size	500–5,000 sq ft
Bedrooms	1–5

Without scaling, the features have very different numerical magnitudes. In models trained with gradient descent, this can create an uneven optimization landscape: the loss function may be steep in some parameter directions and shallow in others.

Standardization can therefore:
- Improve optimization speed for models such as linear regression trained with gradient descent and logistic regression.
- Make regularization behave more consistently across features.
- Reduce numerical difficulties caused by extreme differences in feature magnitudes.
It does not guarantee faster training for every model. Decision trees and random forests, for example, generally do not require standardization.

3. Applying it in Python

In [ ]:
from sklearn.preprocessing import StandardScaler

features = df[["square_feet", "bedrooms"]]

scaler = StandardScaler()
scaled_features = scaler.fit_transform(features)

Fit the scaler on the training set only, then use that same fitted scaler to transform the validation and test sets. This prevents information from those sets from leaking into training.

Key takeaway: The mean centers the data; the standard deviation rescales it. Together, they often improve gradient-based optimization by making numerical feature scales more comparable.